In [1]:
import pandas as pd
import geopandas as gpd

import folium
from folium import Choropleth
from folium.plugins import HeatMap

In [2]:
from IPython.display import IFrame, Markdown, display

def embed_map(m, file_name):    
    m.save(file_name)
    return IFrame(file_name, width='100%', height='500px')

In [3]:
PLATE_BOUNDARIES_DATA = "../datasets/geospatial-learn-course-data/Plate_Boundaries/Plate_Boundaries.shp"

plate_boundaries = gpd.read_file(PLATE_BOUNDARIES_DATA)
plate_boundaries['coordinates'] = (plate_boundaries
                                   .apply(lambda x: [(b,a) for (a,b) in list(x.geometry.coords)], axis=1))
plate_boundaries.drop('geometry', axis=1, inplace=True)

plate_boundaries.head()

,HAZ_PLATES,HAZ_PLAT_1,HAZ_PLAT_2,Shape_Leng,coordinates
0,TRENCH,SERAM TROUGH (ACTIVE),6722,5.843467,"[(-5.444200361999947, 133.6808931800001), (-5...."
1,TRENCH,WETAR THRUST,6722,1.829013,"[(-7.760600482999962, 125.47879802900002), (-7..."
2,TRENCH,TRENCH WEST OF LUZON (MANILA TRENCH) NORTHERN ...,6621,6.743604,"[(19.817899819000047, 120.09999798800004), (19..."
3,TRENCH,BONIN TRENCH,9821,8.329381,"[(26.175899215000072, 143.20620700100005), (26..."
4,TRENCH,NEW GUINEA TRENCH,8001,11.998145,"[(0.41880004000006466, 132.8273013480001), (0...."


In [4]:
EARTHQUAKES_DATA = "../datasets/geospatial-learn-course-data/earthquakes1970-2014.csv"
earthquakes = pd.read_csv(EARTHQUAKES_DATA, parse_dates=["DateTime"])
earthquakes.head()

,DateTime,Latitude,Longitude,Depth,Magnitude,MagType,NbStations,Gap,Distance,RMS,Source,EventID
0,1970-01-04 17:00:40.200,24.139,102.503,31.0,7.5,Ms,90.0,NaN,NaN,0.0,NEI,1.970010e+09
1,1970-01-06 05:35:51.800,-9.628,151.458,8.0,6.2,Ms,85.0,NaN,NaN,0.0,NEI,1.970011e+09
2,1970-01-08 17:12:39.100,-34.741,178.568,179.0,6.1,Mb,59.0,NaN,NaN,0.0,NEI,1.970011e+09
3,1970-01-10 12:07:08.600,6.825,126.737,73.0,6.1,Mb,91.0,NaN,NaN,0.0,NEI,1.970011e+09
4,1970-01-16 08:05:39.000,60.280,-152.660,85.0,6.0,ML,0.0,NaN,NaN,NaN,AK,NaN


In [5]:
m_1 = folium.Map(location=[35,136], tiles='cartodbpositron', zoom_start=5)
for i in range(len(plate_boundaries)):
    folium.PolyLine(locations=plate_boundaries.coordinates.iloc[i], weight=2, color='black').add_to(m_1)

# Add a heatmap to the map
HeatMap(data=earthquakes[['Latitude', 'Longitude']], radius=10).add_to(m_1)

embed_map(m_1, 'im_q1.html')

In [6]:
content = f"""
**Q**: **Given the map above, do earthquakes coincide with plate boundaries?**<br>
**A**: Yes, earthquakes coincide with plate boundaries.
"""
display(Markdown(content))


**Q**: **Given the map above, do earthquakes coincide with plate boundaries?**<br>
**A**: Yes, earthquakes coincide with plate boundaries.


In [7]:
from folium import Circle

m_2 = folium.Map(location=[35,136], tiles='cartodbpositron', zoom_start=5)
for i in range(len(plate_boundaries)):
    folium.PolyLine(locations=plate_boundaries.coordinates.iloc[i], weight=2, color='black').add_to(m_2)

def color_producer(val):
    if val < 50:
        return 'forestgreen'
    if val < 100:
        return 'darkorange'
    return 'darkred'

for i in range(0,len(earthquakes)):
    Circle(
        location=[earthquakes.iloc[i]['Latitude'], earthquakes.iloc[i]['Longitude']],
        radius=20,
        color=color_producer(earthquakes.iloc[i]['Depth'])).add_to(m_2)
    
embed_map(m_2, 'im_q2.html')

In [8]:
content = f"""
**Q**: **Can you detect a relationship between proximity to a plate boundary and earthquake depth? 
Does this pattern hold globally? In Japan?**<br>
**A**: In the northern half of Japan, it does appear that earthquakes closer to plate boundaries tend 
to be shallower (and earthquakes farther from plate boundaries are deeper). This pattern is repeated 
in other locations, such as the western coast of South America. But, it does not hold everywhere 
(for instance, in China, Mongolia, and Russia).
"""
display(Markdown(content))


**Q**: **Can you detect a relationship between proximity to a plate boundary and earthquake depth? 
Does this pattern hold globally? In Japan?**<br>
**A**: In the northern half of Japan, it does appear that earthquakes closer to plate boundaries tend 
to be shallower (and earthquakes farther from plate boundaries are deeper). This pattern is repeated 
in other locations, such as the western coast of South America. But, it does not hold everywhere 
(for instance, in China, Mongolia, and Russia).


In [9]:
# GeoDataFrame with prefecture boundaries
PREFECTURES_DATA = "../datasets/geospatial-learn-course-data/japan-prefecture-boundaries/japan-prefecture-boundaries.shp"
prefectures = gpd.read_file(PREFECTURES_DATA)
prefectures.set_index('prefecture', inplace=True)
prefectures.head()


,geometry
prefecture,
Aichi,"MULTIPOLYGON (((137.09523 34.6533, 137.09546 3..."
Akita,"MULTIPOLYGON (((139.55725 39.2033, 139.55765 3..."
Aomori,"MULTIPOLYGON (((141.3986 40.92472, 141.39806 4..."
Chiba,"MULTIPOLYGON (((139.82488 34.98967, 139.82434 ..."
Ehime,"MULTIPOLYGON (((132.55859 32.91224, 132.55904 ..."


In [10]:
POPULATION_DATA = "../datasets/geospatial-learn-course-data/japan-prefecture-population.csv"

population = pd.read_csv(POPULATION_DATA)
population.set_index('prefecture', inplace=True)

# Calculate area (in square kilometers) of each prefecture
area_sqkm = pd.Series(prefectures.geometry.to_crs(epsg=32654).area / 1e6, name='area_sqkm')
stats = population.join(area_sqkm)

# Add density (per square kilometer) of each prefecture
stats['density'] = stats["population"] / stats["area_sqkm"]
stats.head()

,population,area_sqkm,density
prefecture,,,
Tokyo,12868000,1800.614782,7146.448049
Kanagawa,8943000,2383.038975,3752.771186
Osaka,8801000,1923.151529,4576.342460
Aichi,7418000,5164.400005,1436.372085
Saitama,7130000,3794.036890,1879.264806


In [11]:
# Create a base map
m_3 = folium.Map(location=[35,136], tiles='cartodbpositron', zoom_start=5)

# Create a choropleth map to visualize population density
Choropleth(geo_data=prefectures['geometry'].__geo_interface__,
           data=stats['density'],
           key_on="feature.id",
           fill_color='YlGnBu',
           legend_name='Population density (per square kilometer)'
          ).add_to(m_3)

# View the map
embed_map(m_3, 'im_q3.html')

In [12]:
content = f"""
**Q**: **Which three prefectures have relatively higher density than the others? Are they spread 
throughout the country, or all located in roughly the same geographical region?**<br>
**A**: Tokyo, Kanagawa, and Osaka have the highest population density. All of these prefectures are located 
in central Japan, and Tokyo and Kanagawa are adjacent.
"""
display(Markdown(content))


**Q**: **Which three prefectures have relatively higher density than the others? Are they spread 
throughout the country, or all located in roughly the same geographical region?**<br>
**A**: Tokyo, Kanagawa, and Osaka have the highest population density. All of these prefectures are located 
in central Japan, and Tokyo and Kanagawa are adjacent.


In [13]:
m_4 = folium.Map(location=[35,136], tiles='cartodbpositron', zoom_start=5)

def color_producer(magnitude):
    if magnitude > 6.5:
        return 'red'
    return 'green'

Choropleth(
    geo_data=prefectures['geometry'].__geo_interface__,
    data=stats['density'],
    key_on="feature.id",
    fill_color='BuPu',
    legend_name='Population density (per square kilometer)').add_to(m_4)

for i in range(0,len(earthquakes)):
    folium.Circle(
        location=[earthquakes.iloc[i]['Latitude'], earthquakes.iloc[i]['Longitude']],
        popup=f"{earthquakes.iloc[i]['Magnitude']} ({earthquakes.iloc[i]['DateTime'].year})",
        radius=earthquakes.iloc[i]['Magnitude']**5.5,
        color=color_producer(earthquakes.iloc[i]['Magnitude'])).add_to(m_4)

embed_map(m_4, 'im_q4.html')

In [14]:
content = f"""
**Q**: **Which prefecture do you recommend for extra earthquake reinforcement?**<br>
**A**: While there's no clear, single answer to this question, there are a few reasonable options. 
**Tokyo** is by far the most densely populated prefecture and has also experienced a number of earthquakes. 
**Osaka** is relatively less densely populated, but experienced an earthquake that was relatively stronger 
than those near Tokyo. And, the long coast of **Kanagawa** (in addition to its high density and the historical 
proximity of strong earthquakes) might lead us to worry about the added potential tsunami risk.
"""
display(Markdown(content))


**Q**: **Which prefecture do you recommend for extra earthquake reinforcement?**<br>
**A**: While there's no clear, single answer to this question, there are a few reasonable options. 
**Tokyo** is by far the most densely populated prefecture and has also experienced a number of earthquakes. 
**Osaka** is relatively less densely populated, but experienced an earthquake that was relatively stronger 
than those near Tokyo. And, the long coast of **Kanagawa** (in addition to its high density and the historical 
proximity of strong earthquakes) might lead us to worry about the added potential tsunami risk.
